# NRCS SCAN / SNOTEL — In-Situ Station Network (AWDB REST API)

**What it is.** USDA's Air-Water Database (AWDB): ground stations from **SCAN** (Soil Climate
Analysis Network) and **SNOTEL** (mountain snowpack). Real sensors measuring precip, air/soil
temperature, soil moisture, and snow water equivalent.

| | |
|---|---|
| **Coverage** | United States (SCAN nationwide; SNOTEL in the mountain West) |
| **Cost / key** | **Free · no key** |
| **Docs** | https://wcc.sc.egov.usda.gov/awdbRestApi/swagger-ui/ |

**Why it's in Tracera.** *Ground-truth* observations to validate gridded weather, and (where
sensors report) in-situ soil moisture. There is a **SCAN station right at Ames**, next to our
sample farm.

> **Data reality:** at build time, the AWDB `/data` endpoint reliably served climate elements
> (precip `PREC`, air temp `TAVG`) for the Iowa SCAN stations, but the depth-based soil-moisture
> element (`SMS`) returned no values there. Soil-moisture availability is **station/period
> specific**. For guaranteed gridded soil moisture use **CROP-CASMA / SMAP** (see
> `vegscape_cropcasma.ipynb`) or **OpenET** for water use.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Find the SCAN station nearest the field

In [2]:
import math

AWDB = "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1"

def nearest_scan(lat=LAT, lon=LON):
    stations = requests.get(f"{AWDB}/stations",
                            params={"networkCds": "SCAN", "activeOnly": "true"}, timeout=90).json()
    scan = [s for s in stations if s.get("networkCode") == "SCAN"]
    def dist(s):
        return math.hypot(s["latitude"] - lat, s["longitude"] - lon)
    return min(scan, key=dist)

station = nearest_scan()
TRIPLET = station["stationTriplet"]
print(f"Nearest SCAN station: {station['name']} ({TRIPLET}) "
      f"at ({station['latitude']}, {station['longitude']})")

Nearest SCAN station: Ames (2031:IA:SCAN) at (42.01667, -93.73333)


### 2. Pull daily observations that this station reliably serves (precip, air temp)

In [3]:
def awdb_daily(triplet, elements, start, end):
    r = requests.get(f"{AWDB}/data", params={"stationTriplets": triplet,
        "elements": elements, "duration": "DAILY", "beginDate": start, "endDate": end}, timeout=90)
    r.raise_for_status()
    out = {}
    for station in r.json():
        for el in station.get("data", []):
            code_ = el["stationElement"]["elementCode"]
            unit = el["stationElement"]["storedUnitCode"]
            s = pd.Series({v["date"]: v["value"] for v in el.get("values", [])})
            out[f"{code_}_{unit}"] = s
    df = pd.DataFrame(out)
    df.index = pd.to_datetime(df.index)
    return df

obs = awdb_daily(TRIPLET, "PREC,TAVG", "2023-06-01", "2023-06-10")
print(f"In-situ daily observations at {station['name']}:")
obs

In-situ daily observations at Ames:


,PREC_in,TAVG_degF
2023-06-01,15.58,68.7
2023-06-02,15.58,76.1
2023-06-03,15.58,76.1
2023-06-04,15.60,74.8
2023-06-05,15.60,73.6
2023-06-06,15.60,74.5
2023-06-07,15.63,70.0
2023-06-08,15.63,66.9
2023-06-09,15.63,68.4
2023-06-10,15.63,69.1


### 3. Soil-moisture helper (ready to run where a station reports `SMS`)

In [4]:
def scan_soil_moisture(triplet, start, end):
    """SCAN volumetric soil moisture (%) by depth. Returns empty if the station
    does not serve SMS for the period (common — see the note above)."""
    r = requests.get(f"{AWDB}/data", params={"stationTriplets": triplet, "elements": "SMS",
        "duration": "DAILY", "beginDate": start, "endDate": end}, timeout=90)
    frames = {}
    for station in r.json():
        for el in station.get("data", []):
            depth = el["stationElement"].get("heightDepth")
            frames[f"SMS_depth_{depth}"] = pd.Series({v["date"]: v["value"] for v in el.get("values", [])})
    return pd.DataFrame(frames)

sm = scan_soil_moisture(TRIPLET, "2023-06-01", "2023-06-10")
print("Soil moisture rows returned:", len(sm),
      "(empty here — this station doesn't serve SMS; try CROP-CASMA/SMAP instead)")

Soil moisture rows returned: 0 (empty here — this station doesn't serve SMS; try CROP-CASMA/SMAP instead)


### Notes & how to extend
- **Networks:** set `networkCds="SNOTEL"` for mountain snowpack / **SWE** (snow water
  equivalent) — the driver of Western irrigation water supply.
- **Elements:** `PREC` (precip), `TAVG/TMAX/TMIN` (air temp), `SMS` (soil moisture %),
  `STO` (soil temp), `WTEQ` (SWE), `SNWD` (snow depth). Depth elements use `heightDepths`.
- **Station params are filtered client-side** here because the API returned the full station
  list; for production, cache the SCAN station list once.
- Use as **ground-truth** to bias-correct **gridMET** / **NASA POWER** near a station.